# Scrape used cars from Vucar

Extract the used-car **catalogue** at https://vucar.vn/mua-ban/o-to-cu into a CSV.

Your original notebook opened Chrome and printed page source. This version extracts the records, follows real data pagination, deduplicates listings, and saves after every page. It calls the public `fetchCarListing` data action used by the website, so **no Chrome, Selenium, account, or API key is needed**.

The default run is a **3-page trial**. Change `MAX_PAGES` to `None` to collect the entire catalogue. Data is saved as `vucar_cars.csv` with a companion `vucar_cars.summary.json` recording whether the run is complete.

Verified against the live website on **18 October 2026**. Catalogue records and displayed prices do not establish current availability or a completed sale price.

## Output fields

| CSV column | Website field | Meaning |
|---|---|---|
| `model_year_name` | `display_name` | Full model/year name, e.g. Toyota Innova 2017 |
| `year` | `manufactor_year` | Numeric production/model year as supplied by Vucar |
| `odo_km` | `car_mileage` | Numeric odometer reading in kilometres |
| `car_type` | `body_type` | Body style, such as Sedan, SUV, or MPV |
| `price_vnd` | `car_price` | Full listed price in VND; 418000000 means 418 million VND |

Additional columns: transmission code, location, listing URL, source page, and collection timestamp. Missing values stay blank; a reported zero stays zero. Price is the car's listed price, not the monthly instalment amount.

## 1. Check the runtime

Use Python 3.10+ in Google Colab or Jupyter. The scraper uses Python's standard library and the `curl` command, which is normally available in Colab. Pandas is optional and is used only for the preview. Run the cells in order.

In [ ]:
import shutil
import sys

if sys.version_info < (3, 10):
    raise RuntimeError("Use Python 3.10 or newer.")
if shutil.which("curl") is None:
    raise RuntimeError("Install curl using your operating system's package manager, then rerun.")
print("Runtime ready. Chrome installation is not required.")

## 2. Request helpers and endpoint discovery

Vucar uses a Next.js server action for listing pagination. Its action ID can change when the site is deployed. The code discovers the current ID from the catalogue's public JavaScript files rather than keeping a fixed build-specific ID.

Requests are sequential and spaced apart. HTTP 401, 403, and 429 stop the run; transient network/server errors have limited retries.

In [ ]:
"""Scrape Vucar's used-car catalogue using its public listing data action.

Python 3.10+ and curl are required; Chrome and third-party Python packages are not.
Run: python vucar_scraper.py --max-pages 3
Use --max-pages 0 for the entire catalogue. Output is UTF-8-with-BOM CSV.
"""

from __future__ import annotations

import argparse
import csv
import json
import re
import shutil
import subprocess
import time
import warnings
from datetime import datetime, timezone
from html.parser import HTMLParser
from pathlib import Path
from urllib.parse import urljoin, urlsplit
from urllib.robotparser import RobotFileParser


BASE_URL = "https://vucar.vn"
CATALOGUE_URL = BASE_URL + "/mua-ban/o-to-cu"
COLUMNS = [
    "model_year_name", "year", "odo_km", "car_type", "price_vnd",
    "transmission", "location", "url", "source_page", "scraped_at_utc",
]


# 1. Make ordinary HTTP requests. Do not use shell=True or disable TLS checks.
class CurlClient:
    def __init__(self, delay: float = 1.5, timeout: int = 60):
        self.executable = shutil.which("curl")
        if self.executable is None:
            raise RuntimeError("curl is required. It is normally preinstalled in Google Colab.")
        self.delay = delay
        self.timeout = timeout
        self.last_request = 0.0

    def request(self, url: str, *, headers: dict | None = None, body: str | None = None) -> str:
        if urlsplit(url).scheme != "https" or urlsplit(url).netloc != "vucar.vn":
            raise ValueError("Only public HTTPS URLs on vucar.vn are supported.")
        marker = "\n__VUCAR_HTTP_STATUS__:"
        for attempt in range(3):
            time.sleep(max(0.0, self.last_request + self.delay - time.monotonic()))
            command = [
                self.executable, "--silent", "--show-error", "--compressed",
                "--connect-timeout", "15", "--max-time", str(self.timeout),
                "--write-out", marker + "%{http_code}",
            ]
            for key, value in (headers or {}).items():
                command.extend(["--header", f"{key}: {value}"])
            if body is not None:
                command.extend(["--data-binary", "@-"])
            command.append(url)
            self.last_request = time.monotonic()
            result = subprocess.run(
                command, input=body, capture_output=True, text=True,
                encoding="utf-8", errors="replace", timeout=self.timeout + 10,
            )
            payload, separator, status_text = result.stdout.rpartition(marker)
            status = int(status_text) if separator and status_text.isdigit() else 0
            if result.returncode == 0 and status == 200:
                return payload
            if status in (401, 403, 429):
                raise RuntimeError(
                    f"Vucar returned HTTP {status}; stopping. Already collected rows are saved."
                )
            if attempt < 2 and (result.returncode != 0 or status >= 500):
                time.sleep(2 ** (attempt + 1))
                continue
            detail = result.stderr.strip() or payload[:160].replace("\n", " ")
            raise RuntimeError(f"Request failed: HTTP {status}, curl {result.returncode}: {detail}")
        raise RuntimeError("Request failed.")


class ScriptSources(HTMLParser):
    def __init__(self):
        super().__init__()
        self.urls: list[str] = []

    def handle_starttag(self, tag, attrs):
        attributes = dict(attrs)
        source = attributes.get("src", "")
        if tag == "script" and "/_next/" in source and ".js" in source:
            url = urljoin(BASE_URL, source)
            if urlsplit(url).netloc == "vucar.vn" and url not in self.urls:
                self.urls.append(url)


def discover_listing_action(client: CurlClient) -> tuple[str, int | None]:
    """Find the current action ID in public page scripts; it changes on deployment."""
    robots = RobotFileParser()
    robots.parse(client.request(BASE_URL + "/robots.txt").splitlines())
    if not robots.can_fetch("curl", CATALOGUE_URL):
        raise RuntimeError("The catalogue is disallowed by the current robots.txt.")
    client.delay = max(client.delay, float(robots.crawl_delay("curl") or 0))
    html = client.request(CATALOGUE_URL)
    total_match = re.search(r"Tìm thấy\s*<span\b[^>]*>([\d.,]+)", html)
    catalogue_total = int(re.sub(r"\D", "", total_match.group(1))) if total_match else None
    scripts = ScriptSources()
    scripts.feed(html)
    pattern = re.compile(
        r'''createServerReference\)?\(\s*["']([a-fA-F0-9]{32,64})["'][^;]{0,250}?["']fetchCarListing["']'''
    )
    # Route-specific scripts usually appear near the end. No build hash is hardcoded.
    for url in reversed(scripts.urls):
        if not robots.can_fetch("curl", url):
            continue
        match = pattern.search(client.request(url))
        if match:
            return match.group(1), catalogue_total
    raise RuntimeError(
        "Could not locate fetchCarListing in Vucar's public scripts. "
        "The site may have changed; update the discovery/parser before continuing."
    )

## 3. Parse pages and normalize the requested fields

The response is a stream containing JSON records. The parser reads its car list without executing downloaded JavaScript. Mileage and price come from numeric fields, avoiding locale-dependent parsing of commas, dots, “triệu”, or “tỷ”.

In [ ]:
# 2. Read the public Next.js action's streamed JSON result, without executing JS.
def parse_listing_response(text: str) -> dict:
    for line in text.splitlines():
        if not re.match(r"^[0-9a-fA-F]+:", line):
            continue
        try:
            value = json.loads(line.split(":", 1)[1])
        except json.JSONDecodeError:
            continue
        if isinstance(value, dict) and isinstance(value.get("cars"), list):
            for field in ("total", "pageCount", "limit"):
                if not isinstance(value.get(field), int) or isinstance(value[field], bool):
                    raise ValueError(f"Unexpected listing metadata: {field}.")
            if value["total"] < 0 or value["pageCount"] < 0 or value["limit"] <= 0:
                raise ValueError("Invalid pagination metadata.")
            return value
    raise ValueError(
        "No car-list JSON found in the response. The public action or response format may have changed."
    )


def fetch_listing_page(client: CurlClient, action_id: str, page: int, per_page: int = 8) -> dict:
    parameters = {
        "brandParam": None, "modelParam": None, "priceParam": None,
        "mfDateParam": None, "locationParam": None,
        "page": str(page), "per_page": str(per_page),
        "searchTerm": None, "mileage": None, "carBrands": None,
    }
    response = client.request(
        CATALOGUE_URL,
        headers={
            "Next-Action": action_id,
            "Content-Type": "text/plain;charset=UTF-8",
            "Accept": "text/x-component",
            "Origin": BASE_URL,
        },
        body=json.dumps([parameters], separators=(",", ":")),
    )
    return parse_listing_response(response)


def optional_integer(value, field: str) -> int | None:
    if value is None or value == "" or value == "$undefined":
        return None
    if isinstance(value, bool) or not re.fullmatch(r"\d+(?:\.0+)?", str(value)):
        raise ValueError(f"Unexpected numeric value in {field}: {value!r}")
    return int(str(value).split(".")[0])


def normalize_car(car: dict, page: int, timestamp: str) -> dict:
    name, slug = car.get("display_name"), car.get("slug")
    if not name or not slug or not isinstance(slug, str) or not re.fullmatch(r"[A-Za-z0-9_-]+", slug):
        raise ValueError("A listing is missing a valid display_name or slug.")
    body_type = car.get("body_type")
    if body_type:
        body_type = body_type.upper() if body_type.upper() in {"SUV", "MPV", "CUV"} else body_type.title()
    return {
        "model_year_name": name.strip(),
        # Vucar spells this key 'manufactor_year'.
        "year": optional_integer(car.get("manufactor_year"), "year"),
        "odo_km": optional_integer(car.get("car_mileage"), "odo_km"),
        "car_type": body_type,
        "price_vnd": optional_integer(car.get("car_price"), "price_vnd"),
        "transmission": car.get("car_transmission"),
        "location": car.get("car_location"),
        "url": BASE_URL + "/mua-ban/" + slug,
        "source_page": page,
        "scraped_at_utc": timestamp,
    }

## 4. Collect pages and save progress

Each page is checked for new listing URLs. Repeated pages and unexpected empty results raise an error, while previous completed pages remain saved. A run is marked complete only after the last page and when the unique record count matches the site's reported total.

In [ ]:
# 3. Save after every page, deduplicate by listing URL, and record completeness.
def save_csv(rows: list[dict], output: Path):
    output.parent.mkdir(parents=True, exist_ok=True)
    temporary = output.with_name(output.name + ".tmp")
    with temporary.open("w", newline="", encoding="utf-8-sig") as file:
        writer = csv.DictWriter(file, fieldnames=COLUMNS)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(output)


def scrape_vucar(
    output: str | Path = "vucar_cars.csv", *, max_pages: int | None = None,
    per_page: int = 8, delay: float = 1.5,
) -> list[dict]:
    """Collect the catalogue. max_pages=None means all pages; 3 is a quick trial."""
    if per_page not in (8, 24, 72):
        raise ValueError("per_page must be one of the site's options: 8, 24, 72.")
    if max_pages is not None and (isinstance(max_pages, bool) or not isinstance(max_pages, int) or max_pages < 1):
        raise ValueError("max_pages must be a positive integer or None.")
    if delay < 0:
        raise ValueError("delay must be non-negative.")
    output = Path(output)
    summary_path = output.with_suffix(".summary.json")
    rows: list[dict] = []
    seen_urls: set[str] = set()
    seen_pages: set[tuple] = set()
    timestamp = datetime.now(timezone.utc).isoformat(timespec="seconds")
    summary = {
        "source": CATALOGUE_URL, "scraped_at_utc": timestamp,
        "rows": 0, "pages_fetched": 0, "reported_total": None,
        "reported_page_count": None, "duplicates_skipped": 0,
        "complete": False, "stop_reason": "starting",
    }
    output.parent.mkdir(parents=True, exist_ok=True)
    try:
        client = CurlClient(delay=delay)
        print("Discovering Vucar's current public listing action...", flush=True)
        action_id, catalogue_total = discover_listing_action(client)
        page = 1
        while True:
            data = fetch_listing_page(client, action_id, page, per_page)
            if data["limit"] != per_page:
                raise RuntimeError("Vucar returned a different page size; stopping to avoid skipped records.")
            if data["total"] == 0 and catalogue_total != 0:
                raise RuntimeError(
                    "The data action returned an empty catalogue, but the listing page did not confirm zero cars. "
                    "Try per_page=8; the site's larger page sizes may be unavailable."
                )
            summary["reported_total"] = data["total"]
            summary["reported_page_count"] = data["pageCount"]
            current = [normalize_car(car, page, timestamp) for car in data["cars"]]
            fingerprint = tuple(sorted(row["url"] for row in current))
            if fingerprint in seen_pages:
                raise RuntimeError(f"Page {page} repeated a previous page; stopping to avoid a pagination loop.")
            if not current and data["total"] > 0:
                raise RuntimeError(f"Page {page} was empty despite a nonempty catalogue.")
            seen_pages.add(fingerprint)
            new_count = 0
            for row in current:
                if row["url"] in seen_urls:
                    summary["duplicates_skipped"] += 1
                    continue
                seen_urls.add(row["url"])
                rows.append(row)
                new_count += 1
            if current and new_count == 0:
                raise RuntimeError(f"Page {page} contained no new listing URLs.")
            summary.update(rows=len(rows), pages_fetched=page, stop_reason="in_progress")
            save_csv(rows, output)
            summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
            print(f"Page {page}/{data['pageCount']}: +{new_count} cars; {len(rows)}/{data['total']} unique cars saved", flush=True)
            if page >= data["pageCount"]:
                summary["stop_reason"] = "catalogue_exhausted"
                summary["complete"] = len(rows) == data["total"]
                if not summary["complete"]:
                    warnings.warn("Reached the last page, but unique count differs from Vucar's total. See the summary.")
                break
            if max_pages is not None and page >= max_pages:
                summary["stop_reason"] = "max_pages_reached"
                break
            page += 1
    except BaseException as error:
        summary["stop_reason"] = "interrupted" if isinstance(error, KeyboardInterrupt) else "error"
        summary["error"] = f"{type(error).__name__}: {error}"
        raise
    finally:
        summary["rows"] = len(rows)
        summary["missing_values"] = {
            field: sum(row[field] is None or row[field] == "" for row in rows)
            for field in ("model_year_name", "year", "odo_km", "car_type", "price_vnd")
        }
        # Do not overwrite an existing CSV if discovery failed before any page arrived.
        if summary["pages_fetched"]:
            save_csv(rows, output)
        summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Saved {len(rows)} cars to {output}; complete={summary['complete']}", flush=True)
    return rows

## 5. Run the scraper

Keep `MAX_PAGES = 3` for the first run. Set `MAX_PAGES = None` for all pages, then rerun this cell.

Use `PER_PAGE = 8`. During validation, the site's larger page-size options (24 and 72) returned empty data despite a nonempty catalogue. The code rejects that mismatch. A full run may take several minutes or longer because it makes one sequential data request per page.

The output files are refreshed by each new run. If a run stops partway through, inspect the summary and keep the CSV already saved.

In [ ]:
MAX_PAGES = 3       # Change to None for the entire catalogue.
PER_PAGE = 8        # Verified working page size.
OUTPUT_CSV = "vucar_cars.csv"

rows = scrape_vucar(
    output=OUTPUT_CSV,
    max_pages=MAX_PAGES,
    per_page=PER_PAGE,
    delay=1.5,
)

## 6. Preview the result

In [ ]:
try:
    import pandas as pd
    from IPython.display import display
except ImportError:
    print(json.dumps(rows[:5], ensure_ascii=False, indent=2))
else:
    df = pd.DataFrame(rows, columns=COLUMNS)
    for column in ["year", "odo_km", "price_vnd"]:
        df[column] = pd.array(df[column], dtype="Int64")
    display(df.head(10))
    print(f"Unique listings: {df['url'].nunique()}")
    print("Missing values in requested fields:")
    print(df[["model_year_name", "year", "odo_km", "car_type", "price_vnd"]].isna().sum())

## 7. Check completeness

`complete: false` with `stop_reason: max_pages_reached` is expected for the three-page trial. `reported_total` is the count reported by the catalogue at collection time. Missing body types or other fields are not inferred from model names.

In [ ]:
summary_file = Path(OUTPUT_CSV).with_suffix(".summary.json")
summary = json.loads(summary_file.read_text(encoding="utf-8"))
print(json.dumps(summary, ensure_ascii=False, indent=2))

## 8. Download the CSV in Google Colab

The CSV uses UTF-8 with a BOM so Vietnamese text opens correctly in Excel. In local Jupyter, the file is already in the notebook's current directory.

In [ ]:
try:
    from google.colab import files
except ImportError:
    print("CSV saved at:", Path(OUTPUT_CSV).resolve())
else:
    files.download(OUTPUT_CSV)

## If the website changes

- **No listing action found / no car-list JSON:** the website's internal request format may have changed. Reinspect the public catalogue and update the discovery/parser rather than assuming an empty dataset.
- **Repeated page or false empty result:** the scraper stops and preserves completed pages. Keep the verified page size of 8.
- **HTTP 401/403/429:** the run stops. Do not treat the saved partial CSV as a complete catalogue.

Standalone Python usage, if you also downloaded `vucar_scraper.py`:

```bash
python vucar_scraper.py --max-pages 3 --output vucar_cars.csv
python vucar_scraper.py --max-pages 0 --output vucar_cars.csv
```

Sources inspected: the public catalogue, its linked JavaScript and data response, and https://vucar.vn/robots.txt. No seller contact details are collected.